# 02 — Unified Model Training: MLP, LSTM, Hybrid & Random Forest
Train and compare every integrated model family from one notebook using a
single registry, one split policy, and one centralized training API.

| Architecture | Description |
|---|---|
| **MLP** | Per-timestep features from the last step of the window |
| **LSTM** | Bi-directional LSTM over the full window |
| **Hybrid** | MLP branch (last step) + LSTM branch fused |
| **Random Forest** | Static tabular baseline trained on the same file-level splits |

This notebook is the canonical training entry point.
All model families share the same registry-driven train/val/test split,
which prevents file-level leakage and enables fair comparisons.

## 1. Imports & setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

from library.utils import setup_logging, get_device, get_num_workers
from library.config import BatchConfig
from library.data import prepare_file_registry, assign_splits
from library.data.registry import save_registry_manifest  # A1/A2: manifest persistence
from library.models import run_model_task

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Configuration
Tune this section before running training.

### Paths
- `DATA_DIR`: folder containing parquet simulations.
- `OUTPUT_DIR`: artifact destination (models, plots, scalers, reports).

### Sequence windowing
- `WINDOW_SIZE`: timesteps per sequence sample (larger captures longer dynamics, costs more RAM/time).
- `STRIDE`: spacing between windows (smaller = more overlap and more samples).

### Training controls
- `BATCH_SIZE`: sequence mini-batch size for neural models; lower if you hit memory limits.
- `EPOCHS`: maximum epochs.
- `PATIENCE`: early-stopping patience based on validation loss.

### Dataset scope
- `MAX_RUNS`: optional cap on number of simulation files for quicker experiments.
  Set to `None` for full training; keep this value fixed across experiments for fairness.

### Model selection
- `MODEL_MODES`: list of model backends to run.
- `CUSTOM_MODEL_RUNNERS`: plug-in runners for future custom modes.

### Leakage safety
Do not manually build your own split in downstream cells. Always use
`prepare_file_registry(...)` + `assign_splits(...)` once from this notebook so
all models evaluate on the same held-out files.

In [ ]:
# -------- Paths --------
DATA_DIR = "dataset"
OUTPUT_DIR = "outputs/sequence_models"
MANIFEST_PATH = f"{OUTPUT_DIR}/registry_manifest.json"  # A1/A2: persisted registry + model paths
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# -------- Sequence/window parameters --------
# 2016 = 7 days at 5-minute cadence (7 * 24 * 12).
# Increase for longer temporal context; decrease to reduce memory/time.
WINDOW_SIZE = 2016

# 288 = 1 day at 5-minute cadence.
# Smaller stride creates more overlapping windows and longer training time.
STRIDE = 288

# -------- Training hyperparameters --------
BATCH_SIZE = 512   # Lower this first if you see memory pressure.
EPOCHS = 30        # Max epochs; early stopping can end sooner.
PATIENCE = 7       # Stop after N non-improving validation epochs.

# -------- Data scope --------
# None = all runs. Use an int (e.g., 96) for fast/debug experiments.
# NOTE: this is intentionally only here in notebook 02 — notebook 03
# loads the registry from MANIFEST_PATH instead of re-deriving it, so
# it never re-runs prepare_file_registry() and this constant never
# affects eval (A1 split-leakage fix).
MAX_RUNS = None

# -------- Redundancy reduction (tabular family only) --------
# Thins whichever fault_active value is locally the MAJORITY within each
# *training* file (decided per file -- NOT assumed to be the healthy side;
# measured on this batch via 01_data_exploration.ipynb's "Fault Prevalence"
# chart: ~80% of rows are fault_active=True here, so faulted rows are what
# actually gets thinned). The minority-class rows are always kept in full,
# and the test split is never thinned (see load_split's docstring). No
# effect on mlp/lstm/hybrid, which need contiguous per-file timesteps for
# windowing.
#
# 1 = no thinning. Both RandomForestClassifier (class_weight="balanced") and
# the sequence models' weighted cross-entropy already compensate for class
# imbalance in the loss, so this isn't required for model quality -- it's a
# memory/speed lever. A factor of ~4 on an 80/20 split lands close to 1:1;
# pushing well past that starts manufacturing a new imbalance in the other
# direction rather than just trimming redundancy.
MAJORITY_THIN_FACTOR = 4

# -------- Tabular row caps --------
# "auto" only guarantees the LOADED DATA fits in memory -- it has no idea
# what you do with that data afterward. RandomForestClassifier(n_estimators=200,
# n_jobs=-1) does NOT scale gently: with "auto" resolving to ~23M rows here,
# fitting 200 trees in parallel across every CPU core was enough to get the
# whole kernel process OS-killed (no Python traceback -- that's the signature
# of an out-of-memory SIGKILL, not a raised exception) right as RF training
# started. RF also gets essentially no accuracy benefit from tens of millions
# of rows once class_weight="balanced" + stratified sampling already give it
# good class coverage, so this is a pure cost with no return.
#
# A concrete, conservative cap avoids this regardless of what "auto" would
# otherwise compute from available RAM. Start here; if training completes
# comfortably (watch system memory during the "RANDOM_FOREST" step), you can
# try raising it -- but scale up gradually and watch memory, rather than
# jumping back to "auto"/None, since neither knows about this failure mode.
MAX_TRAIN_ROWS = 1_000_000
MAX_TEST_ROWS  = 1_000_000

# -------- Batched tabular training --------
# A DIFFERENT fix from MAX_TRAIN_ROWS/"auto" above -- those only bound how
# much gets LOADED, not whether fitting a model on it fits in memory.
# RandomForestClassifier.fit() needs its whole training matrix at once no
# matter how it got capped, and with n_jobs parallelism this can still
# collapse the workstation (no Python traceback -- an OS-level OOM kill,
# not a raised exception) even at a modest, already-capped row count.
#
# Setting ROWS_PER_BATCH switches the tabular family to batched training:
# train_entries are grouped into FILES_PER_BATCH-file batches, each loaded
# and used to grow the forest via scikit-learn's warm_start one batch at a
# time -- previously-built trees are untouched, only NEW trees are fit on
# each new batch -- so peak memory is bounded by one batch's data, never
# the whole training set. The requested n_estimators (200 by default) is
# still the FINAL forest size; it's just split across batches automatically.
#
# None = batching disabled (single load + single .fit(), as before). Only
# affects the tabular family and only modes supporting warm_start
# (currently just "random_forest") -- mlp/lstm/hybrid already train in
# batches by construction (memmap-backed windows + DataLoader) and are
# untouched either way.
ROWS_PER_BATCH  = 200_000
FILES_PER_BATCH = 10

# -------- Unified model selection --------
# Tabular modes (estimator-agnostic): "random_forest", "gradient_boosting",
#   "logistic_regression", "ebm" (optional deps: lightgbm, interpret).
# Sequence modes: "mlp", "lstm", "hybrid".
# Unknown modes are skipped unless a custom runner is provided.
MODEL_MODES = ["mlp", "lstm", "hybrid", "random_forest"]

# Map custom mode name -> callable runner(task_name=..., target_col=..., ...)
CUSTOM_MODEL_RUNNERS = {}


In [ ]:
# Load batch configuration and runtime environment details.
cfg = BatchConfig(DATA_DIR)
device = get_device()
n_workers = get_num_workers()

print(f"Device:  {device}")
print(f"Workers: {n_workers}")
print(cfg)


## 3. Build file registry & assign splits

In [ ]:
# No data is loaded here — just file paths and fault-type metadata
registry = prepare_file_registry(DATA_DIR, cfg, max_runs=MAX_RUNS)
assign_splits(registry)

fault_types = sorted({e['fault_type'] for e in registry})
print(f"\nFault types in registry: {fault_types}")


## 4. Feature list

In [ ]:
# Use the centralized feature set shared across all model families.
feature_list = cfg.feature_set("full")
print(f"Total features: {len(feature_list)}")


## 5. Task A — Fault Detection

In [ ]:
detection_details = run_model_task(
    task_name="Fault Detection",
    target_col="fault_active",
    registry=registry,
    feature_cols=feature_list,
    model_modes=MODEL_MODES,
    output_dir=OUTPUT_DIR,
    cfg=cfg,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=device,
    patience=PATIENCE,
    num_workers=n_workers,
    max_train_rows=MAX_TRAIN_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    majority_thin_factor=MAJORITY_THIN_FACTOR,
    rows_per_batch=ROWS_PER_BATCH,
    files_per_batch=FILES_PER_BATCH,
    custom_runners=CUSTOM_MODEL_RUNNERS,
    return_details=True,  # A2: need model_path per mode for the manifest
)
detection_reports = {m: d["report"] for m, d in detection_details.items()}
print("Trained detection modes:", ", ".join(detection_reports))


In [ ]:
from IPython.display import Image, display as ipy_display

for mode in detection_reports:
    p = f"{OUTPUT_DIR}/curves_fault_detection_{mode}.png"
    if pathlib.Path(p).exists():
        print(f"\n{mode.upper()} training curves:")
        ipy_display(Image(p))


## 6. Task B — Fault Classification (multi-class)

In [ ]:
classification_details = run_model_task(
    task_name="Fault Classification",
    target_col="fault_type",
    registry=registry,
    feature_cols=feature_list,
    model_modes=MODEL_MODES,
    output_dir=OUTPUT_DIR,
    cfg=cfg,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=device,
    patience=PATIENCE,
    num_workers=n_workers,
    max_train_rows=MAX_TRAIN_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    majority_thin_factor=MAJORITY_THIN_FACTOR,
    rows_per_batch=ROWS_PER_BATCH,
    files_per_batch=FILES_PER_BATCH,
    custom_runners=CUSTOM_MODEL_RUNNERS,
    return_details=True,  # A2: need model_path per mode for the manifest
)
classification_reports = {m: d["report"] for m, d in classification_details.items()}
print("Trained classification modes:", ", ".join(classification_reports))


In [ ]:
# Per-fault F1 comparison
p = f"{OUTPUT_DIR}/per_fault_f1_fault_classification.png"
if pathlib.Path(p).exists():
    ipy_display(Image(p))


## 7. Confusion matrices

In [ ]:
available_modes = sorted(set(detection_reports) | set(classification_reports))
for task_tag in ["fault_detection", "fault_classification"]:
    for mode in available_modes:
        p = f"{OUTPUT_DIR}/confusion_{task_tag}_{mode}.png"
        if pathlib.Path(p).exists():
            print(f"\n{task_tag} — {mode.upper()}")
            ipy_display(Image(p))

# Importance plots: show all modes that produced one (tree-based and coef_-based)
for task_tag in ["fault_detection", "fault_classification"]:
    for mode in available_modes:
        p = f"{OUTPUT_DIR}/importance_{task_tag}_{mode}.png"
        if pathlib.Path(p).exists():
            print(f"\n{task_tag} — {mode.upper()} feature importance")
            ipy_display(Image(p))


## 8. Advanced evaluations

In [ ]:
from library.evaluation import run_temporal_split

temporal_results = run_temporal_split(
    registry,
    feature_list,
    pathlib.Path(OUTPUT_DIR),
    cfg,
    model_modes=MODEL_MODES,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=device,
    patience=PATIENCE,
    num_workers=n_workers,
    max_train_rows=MAX_TRAIN_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    majority_thin_factor=MAJORITY_THIN_FACTOR,
    rows_per_batch=ROWS_PER_BATCH,
    files_per_batch=FILES_PER_BATCH,
    custom_runners=CUSTOM_MODEL_RUNNERS,
)
if temporal_results:
    display(pd.DataFrame(temporal_results).T)


## 9. Cross-location generalization

In [ ]:
from library.evaluation import run_cross_location

cross_location_results = run_cross_location(
    registry,
    feature_list,
    pathlib.Path(OUTPUT_DIR),
    cfg,
    model_modes=MODEL_MODES,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=device,
    patience=PATIENCE,
    num_workers=n_workers,
    max_train_rows=MAX_TRAIN_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    majority_thin_factor=MAJORITY_THIN_FACTOR,
    rows_per_batch=ROWS_PER_BATCH,
    files_per_batch=FILES_PER_BATCH,
    custom_runners=CUSTOM_MODEL_RUNNERS,
)
if cross_location_results:
    display(pd.DataFrame(cross_location_results))
    for mode in sorted({row["model_mode"] for row in cross_location_results}):
        p = pathlib.Path(OUTPUT_DIR) / f"cross_location_{mode}.png"
        if p.exists():
            print(f"\nCross-location summary — {mode.upper()}")
            ipy_display(Image(str(p)))


## 10. Feature ablation

In [ ]:
from library.evaluation import run_feature_ablation

ablation_results = run_feature_ablation(
    registry,
    pathlib.Path(OUTPUT_DIR),
    cfg,
    model_modes=MODEL_MODES,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=device,
    patience=PATIENCE,
    num_workers=n_workers,
    max_train_rows=MAX_TRAIN_ROWS,
    max_test_rows=MAX_TEST_ROWS,
    majority_thin_factor=MAJORITY_THIN_FACTOR,
    rows_per_batch=ROWS_PER_BATCH,
    files_per_batch=FILES_PER_BATCH,
    custom_runners=CUSTOM_MODEL_RUNNERS,
)
if ablation_results:
    display(pd.DataFrame(ablation_results))
    for mode in sorted({row["model_mode"] for row in ablation_results}):
        p = pathlib.Path(OUTPUT_DIR) / f"feature_ablation_{mode}.png"
        if p.exists():
            print(f"\nFeature ablation — {mode.upper()}")
            ipy_display(Image(str(p)))


## 11. Saved artifacts

In [ ]:
# ── A1 + A2 fix: persist registry (with splits) + model artifact paths ──────
# Notebook 03 loads this manifest instead of re-running prepare_file_registry()
# + assign_splits(), eliminating the split-leakage risk and the hardcoded
# path reconstruction that caused RF models to never appear in eval.
#
# Bug 6 fix: also persist scaler and label-encoder paths so notebook 03's
# eval_pytorch_model() doesn't have to reconstruct them from hardcoded naming
# conventions that break when artifact_prefix is non-default.
model_paths = {
    "fault_active": {m: d.get("model_path") for m, d in detection_details.items()},
    "fault_type":   {m: d.get("model_path") for m, d in classification_details.items()},
}

import pathlib as _pl
scaler_paths = {
    "fault_active": str(_pl.Path(OUTPUT_DIR) / "scaler_fault_detection.pkl"),
    "fault_type":   str(_pl.Path(OUTPUT_DIR) / "scaler_fault_classification.pkl"),
}
le_paths = {
    "fault_type": str(_pl.Path(OUTPUT_DIR) / "label_encoder_fault_classification.pkl"),
}

save_registry_manifest(
    registry,
    MANIFEST_PATH,
    model_paths=model_paths,
    extra={
        "data_dir":     DATA_DIR,
        "max_runs":     MAX_RUNS,
        "majority_thin_factor": MAJORITY_THIN_FACTOR,
        "max_train_rows": MAX_TRAIN_ROWS,
        "max_test_rows": MAX_TEST_ROWS,
        "rows_per_batch": ROWS_PER_BATCH,
        "files_per_batch": FILES_PER_BATCH,
        "scaler_paths": scaler_paths,
        "le_paths":     le_paths,
    },
)
print(f"Registry manifest saved: {MANIFEST_PATH}")
print("  Model paths saved for nb03 to load:")
for task, paths in model_paths.items():
    for mode, p in paths.items():
        print(f"    [{task}] {mode}: {p}")
print("  Scaler paths:", scaler_paths)
print("  LE paths:", le_paths)


In [ ]:
outputs = sorted(pathlib.Path(OUTPUT_DIR).iterdir())
for f in outputs:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<55s}  {size_mb:.2f} MB")
